# Import modules

In [ ]:
import torch
import torch.optim as optim
from pathlib import Path
import logging
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from models.autoencoder import KoopmanAutoencoder

from models.dataloader import create_dataloaders
from models.metrics import Metric
from models.utils import (
    load_checkpoint, 
    load_datasets, 
    load_config, 
    load_acdm,
)

from models.metrics_utils import (
    run_kae_rollout,
    run_exp_kae_rollout,
    compute_vorticity,
    run_diffusion_rollout,
    ke_timeseries,
    ke_timeseries_spatial,
)
from models.visualization import (
    plot_joint_rollout,
    plot_rollout,
    run_full_analysis,
    plot_koopman_modes,
)
from models.rbf import (
    ma_expansion,
)

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# Set plotting style
plt.style.use("seaborn-v0_8-whitegrid")

cmap = sns.color_palette("icefire", as_cmap=True)

# Configuration


In [ ]:
DIMENSION = 128 # 128 256 512 1024
MODEL = "continous" # "discrete" "continous"
TYPE = "linear" # "linear" "mlp" "eigen"
REGIME = "stable" # "full"
CKPT_INDEX = 40

CONFIG_PATH = f"experiment/{REGIME}/{MODEL}_{TYPE}_{DIMENSION}"
# CKPT_PATH = Path(f"./model_outputs_{REGIME}/{MODEL}_{TYPE}_{DIMENSION}/checkpoints_8/epoch_{CKPT_INDEX}.pth") # updated continous mlp 128 full dataset overfitting non re embedding trained only on Re 100
# CKPT_PATH = Path(f"./model_outputs_{REGIME}/{MODEL}_{TYPE}_{DIMENSION}/dummy-kipero5q/checkpoints/epoch_{CKPT_INDEX}.pth") # updated continous mlp 128 full dataset overfitting non re embedding trained only on Re 100
# CKPT_PATH = Path(f"./model_outputs_{REGIME}/{MODEL}_{TYPE}_{DIMENSION}/run-20260121_171152/checkpoints/epoch_{CKPT_INDEX}.pth") # updated continous mlp 128 full dataset overfitting non re embedding trained only on Re 100
# CKPT_PATH = Path(f"./model_outputs_{REGIME}/{MODEL}_{TYPE}_{DIMENSION}/run-20260122_235923/checkpoints/epoch_{CKPT_INDEX}.pth") # updated continous mlp 128 full dataset overfitting non re embedding trained only on Re 100
# CKPT_PATH = Path(f"./model_outputs_{REGIME}/{MODEL}_{TYPE}_{DIMENSION}/run-20260126_171619/checkpoints/epoch_{CKPT_INDEX}.pth") # good one
# CKPT_PATH = Path(f"./model_outputs_{REGIME}/{MODEL}_{TYPE}_{DIMENSION}/run-20260127_092603/checkpoints/epoch_{CKPT_INDEX}.pth") # updated continous mlp 128 full dataset overfitting non re embedding trained only on Re 100
# CKPT_PATH = Path(f"./model_outputs_{REGIME}/{MODEL}_{TYPE}_{DIMENSION}/run-20260417_152934/checkpoints/epoch_{CKPT_INDEX}.pth") # updated continous mlp 128 full dataset overfitting non re embedding trained only on Re 100
# CKPT_PATH = Path(f"./model_outputs_{REGIME}/{MODEL}_{TYPE}_{DIMENSION}/run-20260326_092000/checkpoints/epoch_{CKPT_INDEX}.pth") # updated continous mlp 128 full dataset overfitting non re embedding trained only on Re 100
CKPT_PATH = Path(f"./model_outputs_{REGIME}/{MODEL}_{TYPE}_{DIMENSION}/checkpoints/epoch_{CKPT_INDEX}.pth") # updated continous mlp 128 full dataset overfitting non re embedding trained only on Re 100


INITIAL_SAMPLE_INDEX = 0
ROLL_OUT_STEPS = 58
FRAME_STRIDE = 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

channels = ["v_x", "v_y", "p", "rho"] if REGIME == "tra" else ["v_x", "v_y", "p"]

In [ ]:
cfg = load_config(CONFIG_PATH)
train_dataset, val_dataset, test_dataset = load_datasets(cfg)
train_loader, val_loader, test_loader = create_dataloaders(
    train_dataset, val_dataset, test_dataset, cfg.training
)

#  Load Model 


In [ ]:
model = KoopmanAutoencoder(
    data_variables=cfg.data.variables,
    input_frames=cfg.data.input_sequence_length,
    height=cfg.model.height,
    width=cfg.model.width,
    latent_dim=cfg.model.latent_dim,
    cond_embedding_dim=cfg.model.cond_embedding_dim,
    cond_type=cfg.model.cond_type,
    operator_mode=cfg.model.operator_mode,
    hidden_dims=cfg.model.hidden_dims,
    transformer_config=cfg.model.transformer,
    use_checkpoint=cfg.training.use_checkpoint,
    predict_cond=cfg.model.predict_cond,
    cond_grad_enabled=cfg.model.cond_grad_enabled,
    disturb_std=None,
    is_continuous=cfg.model.is_continuous,
    rank=cfg.model.rank,
    spectral=cfg.model.spectral,
    use_attention=cfg.model.use_attention,
    cond_expansion_type=cfg.data.selection_param,
    **cfg.model.conv_kwargs,
).to(DEVICE)

print(f"continous model: {model.koopman_operator.is_continuous}")
# if cfg.data.subsample == 1:
#     model.koopman_operator.dt_train = 0.05
# elif cfg.data.subsample == 4:
#     model.koopman_operator.dt_train = 0.2
# elif cfg.data.subsample == 3:
#     model.koopman_operator.dt_train = 0.15
print(f"model.koopman_operator.dt_train: {model.koopman_operator.dt_train}")

model_params = model.parameters()
optimizer = optim.Adam(model_params, lr=cfg.lr_scheduler.lr)
if CKPT_PATH is not None:
    print(f"Loading from checkpoint: {CKPT_PATH}")
    model, _, _, _ = load_checkpoint(CKPT_PATH, model=model, optimizer=optimizer, strict=True)
model.eval()

#  Visualize


In [ ]:
# input_seq, ground_truth, metadata = test_dataset[INITIAL_SAMPLE_INDEX, ROLL_OUT_STEPS]
input_seq_train, ground_truth_train, metadata_train = train_dataset[INITIAL_SAMPLE_INDEX, ROLL_OUT_STEPS]

input_seq_train["obstacle_mask"] = metadata_train["obstacle_mask"][0].repeat(*input_seq_train.batch_size, 1, 1)
input_seq_train["cond_input"] = metadata_train["cond_input"][0].repeat(*input_seq_train.batch_size)

input_seq, ground_truth, metadata = val_dataset[INITIAL_SAMPLE_INDEX, ROLL_OUT_STEPS]
input_seq["obstacle_mask"] = metadata["obstacle_mask"][0].repeat(*input_seq.batch_size, 1, 1)
input_seq["cond_input"] = metadata["cond_input"][0].repeat(*input_seq.batch_size)

total_predicted_seq = run_kae_rollout(
    model, 
    # train_dataset.apply_mask(input_seq), 
    input_seq, 
    ROLL_OUT_STEPS, 
    return_xpreds=False,
    )
total_predicted_seq_train = run_kae_rollout(
    model, 
    input_seq_train, 
    ROLL_OUT_STEPS, 
    return_xpreds=False,
    )

predicted_seq = total_predicted_seq.x_preds.squeeze(0).cpu()
predicted_seq_train = total_predicted_seq_train.x_preds.squeeze(0).cpu()

total_predicted_seq_exp = run_exp_kae_rollout(
    model, 
    input_seq, 
    ROLL_OUT_STEPS, 
    return_xpreds=False,
    )
predicted_seq_exp = total_predicted_seq_exp.x_preds.squeeze(0).cpu()

logger.info(model.timings)


#  Plot Results 


In [ ]:
ground_truth = train_loader.denormalize(ground_truth)
predicted_seq = train_loader.denormalize(predicted_seq.detach())
predicted_seq_exp = train_loader.denormalize(predicted_seq_exp.detach())

initial_state = train_loader.denormalize(input_seq)[-1:, ]
initial_state = initial_state.select(*channels)
# predicted_seq_with_init = torch.cat([initial_state[0].unsqueeze(0), total_predicted_seq.x_recon.cpu(), predicted_seq], dim=0)
predicted_seq_with_init = torch.cat([initial_state, predicted_seq], dim=0)
predicted_seq_exp_with_init = torch.cat([initial_state, predicted_seq_exp], dim=0)
ground_truth_with_init = torch.cat([initial_state, ground_truth.select(*channels)], dim=0)

ground_truth["vort"] = compute_vorticity(ground_truth["v_x"], ground_truth["v_y"])
ground_truth_with_init["vort"] = compute_vorticity(ground_truth_with_init["v_x"], ground_truth_with_init["v_y"])

predicted_seq["vort"] = compute_vorticity(predicted_seq["v_x"], predicted_seq["v_y"])
predicted_seq_with_init["vort"] = compute_vorticity(predicted_seq_with_init["v_x"], predicted_seq_with_init["v_y"])

predicted_seq_exp_with_init["vort"] = compute_vorticity(predicted_seq_exp_with_init["v_x"], predicted_seq_exp_with_init["v_y"])


for var in predicted_seq.keys():
    plot_rollout(ground_truth_with_init.cpu(), predicted_seq_with_init.cpu(), variable_name=var, frame_stride=FRAME_STRIDE) 


In [ ]:
# for var in predicted_seq_exp_with_init.keys():
#     plot_rollout(ground_truth_with_init.cpu(), predicted_seq_exp_with_init.cpu(), variable_name=var, frame_stride=FRAME_STRIDE) 


In [ ]:
# for var in predicted_seq_with_init.keys():
#     plot_joint_rollout(
#         ground_truth_with_init.cpu(),
#         predicted_seq_with_init.cpu(),
#         predicted_seq_exp_with_init.cpu(), # Pass the correctly transposed TensorDict
#         variable_name=var,
#         frame_stride=FRAME_STRIDE,
#         re=metadata["cond_input"][0],
#         pad = [0.022, 0.027, 0.015, 0.02], # left, right, bottom, top
#         model_name = "Exp",
#         show_intermodel_diff=True
#     )

In [ ]:

ground_truth_train = train_loader.denormalize(ground_truth_train)
predicted_seq_train = train_loader.denormalize(predicted_seq_train)
initial_state_train = train_loader.denormalize(input_seq_train)[-1:, ]
initial_state_train = initial_state_train.select(*channels)

predicted_seq_with_init_train = torch.cat([initial_state_train, predicted_seq_train], dim=0)
ground_truth_with_init_train = torch.cat([initial_state_train, ground_truth_train], dim=0)

ground_truth_train["vort"] = compute_vorticity(ground_truth_train["v_x"], ground_truth_train["v_y"])
ground_truth_with_init_train["vort"] = compute_vorticity(ground_truth_with_init_train["v_x"], ground_truth_with_init_train["v_y"])

predicted_seq_train["vort"] = compute_vorticity(predicted_seq_train["v_x"], predicted_seq_train["v_y"])
predicted_seq_with_init_train["vort"] = compute_vorticity(predicted_seq_with_init_train["v_x"], predicted_seq_with_init_train["v_y"])

for var in predicted_seq.keys():
    plot_rollout(ground_truth_with_init_train.cpu(), predicted_seq_with_init_train.cpu(), variable_name=var, frame_stride=FRAME_STRIDE)

In [ ]:
ckpt_diff_0 = "/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/pretrained_models/models_inc/128_acdm-r20_00/Model_full.pth"

ckpt_diff_1 = "/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/pretrained_models/models_inc/128_acdm-r20_00/Model.pth"
ckpt_diff_2 = "/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/pretrained_models/models_inc/128_acdm-r20_01/Model.pth"
ckpt_diff_3 = "/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/pretrained_models/models_inc/128_acdm-r20_02/Model.pth"
ckpt_diff_4 = "/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/pretrained_models/models_inc/128_acdm-r20_ncn_00/Model.pth"
ckpt_diff_5 = "/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/pretrained_models/models_inc/128_acdm-r20_ncn_01/Model.pth"
ckpt_diff_6 = "/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/pretrained_models/models_inc/128_acdm-r20_ncn_02/Model.pth"

ckpt_diff_7 = "/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/pretrained_models/models_tra/128_acdm-r20_02/Model.pth"

ckpt = ckpt_diff_7
# model_diff = load_acdm(ckpt = ckpt, device=DEVICE, rollout_steps=ROLL_OUT_STEPS, subsample=cfg.data.subsample, regime = REGIME)
model_diff = load_acdm(ckpt = ckpt, device=DEVICE, rollout_steps=ROLL_OUT_STEPS, subsample=2, regime = REGIME)

In [ ]:
pred_td = run_diffusion_rollout(model_diff, input_seq, metadata, ROLL_OUT_STEPS, val_dataset, regime=REGIME)
pred_td = train_loader.denormalize(pred_td)

diff_pred_with_init = torch.cat([initial_state, pred_td], dim=0)
pred_td["vort"] = compute_vorticity(pred_td["v_x"], pred_td["v_y"])
pred_td["KE"] = ke_timeseries_spatial(pred_td)

diff_pred_with_init["vort"] = compute_vorticity(diff_pred_with_init["v_x"], diff_pred_with_init["v_y"])
ground_truth_with_init["KE"] = ke_timeseries_spatial(ground_truth_with_init)
predicted_seq_with_init["KE"] = ke_timeseries_spatial(predicted_seq_with_init)
diff_pred_with_init["KE"] = ke_timeseries_spatial(diff_pred_with_init)

pred_td_train = run_diffusion_rollout(model_diff, input_seq_train, metadata_train, ROLL_OUT_STEPS, val_dataset, regime=REGIME)
pred_td_train = train_loader.denormalize(pred_td_train)

diff_pred_with_init_train = torch.cat([initial_state_train, pred_td_train], dim=0)
pred_td_train["vort"] = compute_vorticity(pred_td_train["v_x"], pred_td_train["v_y"])
diff_pred_with_init_train["vort"] = compute_vorticity(diff_pred_with_init_train["v_x"], diff_pred_with_init_train["v_y"])
ground_truth_with_init_train["KE"] = ke_timeseries_spatial(ground_truth_with_init_train)
predicted_seq_with_init_train["KE"] = ke_timeseries_spatial(predicted_seq_with_init_train)
diff_pred_with_init_train["KE"] = ke_timeseries_spatial(diff_pred_with_init_train)


In [ ]:
# for var in pred_td.keys():
#     plot_joint_rollout(
#         ground_truth_with_init.cpu(),
#         predicted_seq_with_init.cpu(),
#         diff_pred_with_init.cpu(), # Pass the correctly transposed TensorDict
#         variable_name=var,
#         frame_stride=FRAME_STRIDE,
#         re=metadata["cond_input"][0],
#         pad = [0.022, 0.027, 0.015, 0.02], # left, right, bottom, top
#         model_name = "ACDM",
#     )
plot_joint_rollout(
    ground_truth_with_init.cpu(),
    predicted_seq_with_init.cpu(),
    diff_pred_with_init.cpu(), # Pass the correctly transposed TensorDict
    variable_name="v_x",
    frame_stride=FRAME_STRIDE,
    re=metadata["cond_input"][0],
    pad = [0.022, 0.027, 0.015, 0.02], # left, right, bottom, top
    model_name = "ACDM",
)

In [ ]:
# for var in pred_td.keys():
#     plot_joint_rollout(
#         ground_truth_with_init_train.cpu(),
#         predicted_seq_with_init_train.cpu(),
#         diff_pred_with_init_train.cpu(), # Pass the correctly transposed TensorDict
#         variable_name=var,
#         frame_stride=FRAME_STRIDE,
#         re=metadata_train["cond_input"][0],
#         pad = [0.022, 0.027, 0.015, 0.02], # left, right, bottom, top
#         model_name = "ACDM",
#     )

plot_joint_rollout(
    ground_truth_with_init_train.cpu(),
    predicted_seq_with_init_train.cpu(),
    diff_pred_with_init_train.cpu(), # Pass the correctly transposed TensorDict
    variable_name="v_x",
    frame_stride=FRAME_STRIDE,
    re=metadata_train["cond_input"][0],
    pad = [0.022, 0.027, 0.015, 0.02], # left, right, bottom, top
    model_name = "ACDM",
)

In [ ]:
# --- Compute KE for test/predicted data ---
ke_gt = ke_timeseries(ground_truth_with_init)
ke_kae = ke_timeseries(predicted_seq_with_init)
ke_diff = ke_timeseries(diff_pred_with_init)

# Optionally normalize (currently just identity)
# ke_gt_norm = ke_gt / ke_gt
# ke_kae_norm = ke_kae / ke_gt
# ke_diff_norm = ke_diff / ke_gt

ke_gt_norm = ke_gt
ke_kae_norm = ke_kae
ke_diff_norm = ke_diff

# --- Compute KE for training data ---
ke_gt_train = ke_timeseries(ground_truth_with_init_train)
ke_kae_train = ke_timeseries(predicted_seq_with_init_train)
ke_diff_train = ke_timeseries(diff_pred_with_init_train)

# ke_gt_norm_train = ke_gt_train / ke_gt_train
# ke_kae_norm_train = ke_kae_train / ke_gt_train
# ke_diff_norm_train = ke_diff_train / ke_gt_train

ke_gt_norm_train = ke_gt_train
ke_kae_norm_train = ke_kae_train
ke_diff_norm_train = ke_diff_train

# --- Determine shared y-axis limits ---
all_ke = np.concatenate([
    ke_gt_norm, ke_kae_norm, ke_diff_norm,
    ke_gt_norm_train, ke_kae_norm_train, ke_diff_norm_train
])
ymin, ymax = all_ke.min(), all_ke.max()

# --- Compute means of ground truth ---
mean_gt_test = ke_gt_norm[:].mean().item()
mean_gt_train = ke_gt_norm_train[:].mean().item()

# --- Create side-by-side subplots with shared y-axis ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# --- Left subplot: Test / Prediction data ---
axes[0].plot(ke_kae_norm, label='kae', color='tab:blue')
axes[0].plot(ke_diff_norm, label='diff', color='tab:orange')
axes[0].plot(ke_gt_norm, label='gt', color='tab:green')
axes[0].axhline(mean_gt_test, color='tab:green', linestyle='--', linewidth=1.5, label='gt mean')
axes[0].set_title(f"Test KE norm (start: {INITIAL_SAMPLE_INDEX}, dim: {DIMENSION})")
axes[0].set_ylim(ymin, ymax)
axes[0].grid(True, linestyle=':')

# --- Right subplot: Training data ---
axes[1].plot(ke_kae_norm_train, color='tab:blue')
axes[1].plot(ke_diff_norm_train, color='tab:orange')
axes[1].plot(ke_gt_norm_train, color='tab:green')
axes[1].axhline(mean_gt_train, color='tab:green', linestyle='--', linewidth=1.5, label='gt mean')
axes[1].set_title(f"Train KE norm (start: {INITIAL_SAMPLE_INDEX}, dim: {DIMENSION})")
axes[1].set_ylim(ymin, ymax)
axes[1].grid(True, linestyle=':')

# --- Single combined legend ---
lines_labels = [
    axes[0].get_lines()[0],  # kae
    axes[0].get_lines()[1],  # diff
    axes[0].get_lines()[2],  # gt
    axes[0].get_lines()[-1]  # gt mean line
]
labels = [line.get_label() for line in lines_labels]
fig.legend(lines_labels, labels, loc='upper center', ncol=4, fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.92])  # leave space for the legend
plt.show()

In [ ]:
def run_metric_analysis(
    gt_train, diff_pred_train, kae_pred_train, metadata_train,
    gt_val, diff_pred_val, kae_val, metadata_val,
    cfg, 
    subsample=1
):
    """
    Computes LSIM metrics for train/val sets and generates a comparison plot with bounds.
    """
    # 1. Initialize Metric
    metric = Metric(
        mode="LSIM",
        variable_mode="all",
        variable_name=None,
    )

    # 2. Compute Metrics
    # Helper to handle the unsqueeze/transpose repetitive logic
    def compute(gt, pred):
        return metric(
            gt[::subsample].unsqueeze(0).transpose(-1, -2),
            pred[::subsample].unsqueeze(0).transpose(-1, -2)
        ).squeeze().cpu().detach().numpy()

    diff_vals_train = compute(gt_train, diff_pred_train)
    kae_vals_train = compute(gt_train, kae_pred_train)
    diff_vals_val = compute(gt_val, diff_pred_val)
    kae_vals_val = compute(gt_val, kae_val)

    # 3. Print Summary
    print(f"\n--- Metric Summary ({cfg.metric.mode}) ---")
    print(f"Train - Diffusion: {diff_vals_train.mean():.6f} | KAE: {kae_vals_train.mean():.6f}")
    print(f"Val   - Diffusion: {diff_vals_val.mean():.6f} | KAE: {kae_vals_val.mean():.6f}")

    # 4. Plotting logic with Confidence Bounds
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    def _plot_sub(ax, kae, diff, meta, title_prefix):
        steps = np.arange(len(kae))
        
        # Plot KAE
        ax.plot(steps, kae, label=f"KAE (mean: {kae.mean():.2f})", color="C0", lw=2)
        ax.fill_between(steps, kae - kae.std(), kae + kae.std(), color="C0", alpha=0.15)
        ax.axhline(kae.mean(), linestyle="--", color="C0", alpha=0.5)

        # Plot Diffusion
        ax.plot(steps, diff, label=f"Diffusion (mean: {diff.mean():.2f})", color="C1", lw=2)
        ax.fill_between(steps, diff - diff.std(), diff + diff.std(), color="C1", alpha=0.15)
        ax.axhline(diff.mean(), linestyle="--", color="C1", alpha=0.5)

        ax.set_title(f"{title_prefix}, Ma = {meta['cond_input'][0]:.2f}")
        ax.set_xlabel("Time step")
        ax.legend(fontsize='small')

    _plot_sub(axes[0], kae_vals_train, diff_vals_train, metadata_train, "Train")
    _plot_sub(axes[1], kae_vals_val, diff_vals_val, metadata_val, "Validation")

    axes[0].set_ylabel("LSIM")
    fig.suptitle("LSIM Distance Temporal Evolution", fontsize=14)
    plt.tight_layout()

    return {
        "diff_train": diff_vals_train,
        "kae_train": kae_vals_train,
        "diff_val": diff_vals_val,
        "kae_val": kae_vals_val,
        "fig": fig
    }

results = run_metric_analysis(
    ground_truth_with_init_train, diff_pred_with_init_train, predicted_seq_with_init_train, metadata_train,
    ground_truth_with_init, diff_pred_with_init, predicted_seq_with_init, metadata,
    cfg, subsample=cfg.data.subsample
)

# Compute metrics


In [ ]:
z_preds = total_predicted_seq.z_preds.cpu().detach()[:, :300]
if z_preds.ndim > 2:
    z_preds = z_preds.squeeze()


# for i in range(len(z_preds)):
#     print(f"latent predictions at step {i}: {z_preds[i]}")

norms_ratio = torch.norm(z_preds.cpu().detach()[1:]/z_preds.cpu().detach()[:-1], dim=-1).cpu().detach().numpy()
z_preds_cpu = z_preds

plt.plot(norms_ratio, 'x')
plt.show()

latent_norms = torch.norm(z_preds_cpu, dim=-1)
# latent_norms = z_preds_cpu[:, -1]
latent_mean = latent_norms.mean()
latent_std = latent_norms.std()

window_size = 25
latent_norms_series = pd.Series(latent_norms.numpy())

smoothed_norms = latent_norms_series.rolling(
    window=window_size,
    center=True,
    min_periods=1
).mean()

fig, ax = plt.subplots(figsize=(12, 7))

time_steps = range(len(latent_norms))
ax.plot(
    time_steps,
    latent_norms.numpy(),
    label='L2 Norm (Raw)',
    color='c',
    linewidth=1.5,
    alpha=0.6
)

ax.plot(
    time_steps,
    smoothed_norms,
    label=f'Smoothed (window={window_size})',
    color='orange',
    linewidth=2.5
)

ax.axhline(latent_mean.item(), color='r', linestyle='--', label='Mean')
upper_bound = latent_mean + 2 * latent_std
lower_bound = latent_mean - 2 * latent_std
ax.fill_between(
    time_steps,
    lower_bound.item(),
    upper_bound.item(),
    color='gray',
    alpha=0.3,
    label='± 2 STD'
)

ax.set_title("L2 Norm of Latent Space Vectors Over Time (with Smoothing)")
ax.set_xlabel("Time Step")
ax.set_ylabel("L2 Norm")
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend() # Add a legend to identify all lines

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# MLP & FiLM-Augmented Koopman Operator Analysis
# =============================================================================

# --- Imports -----------------------------------------------------------------
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

# Ensure deterministic evaluation
torch.set_grad_enabled(False)
model.eval()
# ma_expansion.eval()  # Assuming ma_expansion is defined elsewhere

# =============================================================================
# Helper Functions
# =============================================================================

def get_mlp_jacobian(model, z0, re, device="cuda"):
    """
    Compute the Jacobian of the MLP continuous-time dynamics f(z, re)
    at a single latent state z0.
    """
    z0_t = torch.as_tensor(z0, dtype=torch.float32, device=device).requires_grad_(True)
    re_t = torch.tensor([re], dtype=torch.float32, device=device).detach()

    def f(z):
        return model.koopman_operator.dynamics(
            z.unsqueeze(0),  # Add batch dim
            cond=re_t,       # Fixed conditioning
            dt=0.1
        ).squeeze(0)       # Remove batch dim

    J = torch.func.jacrev(f)(z0_t)
    return J.detach().cpu().numpy()


def get_true_operator(model, re_value, ma_embed, device="cuda"):
    """
    Returns the effective linear Koopman operator for a fixed conditioning value.
    """
    koop = model.koopman_operator.dynamics
    re = torch.tensor([[re_value]], dtype=torch.float32, device=device)
    A = koop._get_effective_linear_map(ma_embed(re))
    return A.cpu().numpy()


def truncated_exp_stability(lam, h, p):
    """
    Returns True if truncated Taylor series of e^(h*lam) up to p terms satisfies |series| ≤ 1
    """
    series_sum = sum((h * lam) ** k / math.factorial(k) for k in range(p + 1))
    return abs(series_sum) <= 1


def plot_eigenvalues(eigenvalues, title, dt=1.0):
    """
    Scatter plot of eigenvalues in the complex plane
    """
    plt.figure(figsize=(8, 6))
    sc = plt.scatter(
        eigenvalues.real,
        eigenvalues.imag,
        c=np.abs(eigenvalues),
        cmap="magma",
        edgecolor="k",
        linewidth=0.5,
        zorder=3
    )
    plt.gca().set_aspect("equal", adjustable="box")
    plt.axhline(0, color="lightgray", lw=1)
    plt.axvline(0, color="lightgray", lw=1)
    plt.title(title, fontsize=14)
    plt.xlabel("Re(λ)")
    plt.ylabel("Im(λ)")
    plt.colorbar(sc, label="|λ|")
    plt.show()


# =============================================================================
# 1. MLP Jacobian Analysis
# =============================================================================

latent_dim = model.latent_dim
z_preds = total_predicted_seq_train.z_preds.detach().cpu()
z0_sample = z_preds[0, 0]  # Single latent vector

J_mlp = get_mlp_jacobian(model, z0_sample, re=0.6)
eigenvalues_mlp = np.linalg.eigvals(J_mlp)
num_pos_real_mlp = np.sum(eigenvalues_mlp.real > 0)

print(f"MLP Jacobian at z0: {num_pos_real_mlp}/{len(eigenvalues_mlp)} eigenvalues with Re(λ) > 0")
plot_eigenvalues(eigenvalues_mlp, "Eigenvalue Spectrum: MLP Jacobian at z₀")


# =============================================================================
# 2. FiLM-Augmented Koopman Operator Analysis
# =============================================================================

ma_embed = ma_expansion(128)  # Assuming ma_expansion is callable
dt = 0.1

A_true = get_true_operator(model, re_value=0.5, ma_embed=ma_embed)
eigenvalues_film, eigenvectors_film = np.linalg.eig(A_true)
eigenvalues_film_scaled = eigenvalues_film * dt

print(f"Eigenvalues (scaled by dt={dt}):\n{eigenvalues_film_scaled}\n")

# Stability check using truncated exponential
stable_flags = np.array([truncated_exp_stability(lam, h=dt, p=15) for lam in eigenvalues_film])
num_stable = stable_flags.sum()
num_unstable = len(eigenvalues_film) - num_stable
num_pos_real = np.sum(eigenvalues_film.real > 0)

print(f"FiLM-Augmented (dt={dt}): {num_stable}/{len(eigenvalues_film)} eigenvalues are stable (|trunc_exp| ≤ 1)")
print(f"{num_unstable} eigenvalues violate the stability condition")
print(f"{num_pos_real} eigenvalues with Re(λ) > 0\n")

plot_eigenvalues(
    eigenvalues_film_scaled,
    title=f"Eigenvalue Spectrum: FiLM-Augmented Koopman Operator (Scaled by dt={dt})"
)

In [ ]:
plot_koopman_modes(
    model=model.cpu(),
    eigenvalues_film=eigenvalues_film,
    num_modes=10,
    device="cpu",
    channel_titles=channels,
)

In [ ]:
# output_folder = "results"
# # # Paths
# # model_to_compare = "continous_linear_128"
# # regime = "lowRey"
# # transit = "full"
# # gt_path = f"/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/src/results/sampling/{regime}/groundTruth.dict"
# # # kae_path = f"/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/src/results/sampling/{regime}/{model_to_compare}.npz"
# # kae_path = f"/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/src/results/sampling/{regime}/{model_to_compare}_{transit}.npz"
# # # acdm_path = f"/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/src/results/sampling/{regime}/acdm-full.npz"
# # acdm_path = f"/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/src/results/sampling/{regime}/acdm-{transit}.npz"

# MODEL = "continous"
# TYPE = "linear"
# DIMENSION = 128
# REGIME = "tra"
# model_to_compare = f"{MODEL}_{TYPE}_{DIMENSION}"
# dataset = "interp"
# gt_path = f"/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/src/results/sampling/{dataset}/groundTruth.dict"
# kae_path = f"/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/src/results/sampling/{dataset}/{MODEL}_{TYPE}_{DIMENSION}_{REGIME}.npz"
# acdm_path = f"/home/rg625/mnt/ocean_forecasting/autoreg_pde_diffusion/src/results/sampling/{dataset}/acdm-r20.npz"


# # Visualization parameters
# time_idx = -1
# channel_idx = 0  # v_x (example variable to visualize)
# sequence_idx = 0  # first sequence



# print("Loading KAE training dataset to extract obstacle mask...")
# try:
#     cfg = load_config(f"experiment/{REGIME}/{model_to_compare}.yaml")
#     kae_train_dataset, _, _ = load_datasets(cfg)
#     kae_mask = kae_train_dataset.obstacle_mask.cpu().numpy()  # (64, 128)
#     kae_mask_transposed = kae_mask.T  # (128, 64)
#     print(kae_mask_transposed)

#     print("Successfully loaded and applied KAE obstacle mask (only first 3 vars masked).")

# except Exception as e:
#     print(f"!!! WARNING: Could not load KAE mask: {e}")
#     print("!!! Masking skipped, using identity mask.")
    
# results = run_full_analysis(
#     gt_path=gt_path,
#     kae_path=kae_path,
#     acdm_path=acdm_path,
#     output_folder=output_folder,
#     obstacle_mask=kae_mask_transposed,  # or None
#     sequence_idx=0,  # first sequence for visualization
#     time_idx=-1,     # last time step for visualization
#     channel_idx=0,   # v_x for visualization
#     model_label=model_to_compare,
#     dataset_label=dataset,
#     regime=REGIME
# )
# gt_masked_all   = results["gt_masked_all"]
# kae_masked_all  = results["kae_masked_all"]
# acdm_masked_all = results["acdm_masked_all"]
# kae_mse         = results["kae_mse"]
# acdm_mse        = results["acdm_mse"]
# Mach_numbers    = results["Mach_numbers"]

# print("Masked MSE KAE (first 5 samples, first field):", kae_mse[:5, 0])
# print("Masked MSE ACDM (first 5 samples, first field):", acdm_mse[:5, 0])

In [ ]:
# from tqdm import tqdm

# all_input_seqs = []
# all_ground_truths = []
# all_predicted_seqs = []

# for idx in tqdm(range(len(val_dataset)), desc="Running val rollouts"):
#     input_seq, ground_truth, metadata = val_dataset[idx, ROLL_OUT_STEPS]

#     # Attach obstacle mask & Re_input
#     input_seq["obstacle_mask"] = metadata["obstacle_mask"][0].repeat(*input_seq.batch_size, 1, 1)
#     input_seq["cond_input"] = metadata["cond_input"][0].repeat(*input_seq.batch_size)

#     # Apply mask for model if needed
#     masked_input = train_dataset.apply_mask(input_seq)

#     # Run KAE rollout
#     total_predicted_seq = run_kae_rollout(
#         model,
#         masked_input,
#         ROLL_OUT_STEPS,
#         return_xpreds=False,
#     )

#     predicted_seq = total_predicted_seq.x_preds.squeeze(0).cpu()
    
#     # Store sequences
#     all_input_seqs.append(train_loader.denormalize(input_seq))
#     all_ground_truths.append(ground_truth)
#     all_predicted_seqs.append(train_loader.denormalize(predicted_seq))

# # Optional: convert to evaluation-ready NumPy arrays
# eval_arrays = []
# for input_seq_td, pred_td in zip(all_input_seqs, all_predicted_seqs):
#     arr = tensordict_to_eval_array_with_cond(input_seq_td, pred_td)
#     eval_arrays.append(arr)

# # Combine into a single array if desired: shape (num_samples, T, C+1, H, W)
# eval_arrays = np.concatenate(eval_arrays, axis=0)
# print("All validation sequences shape:", eval_arrays.shape)

# import os

# out_folder = "results/val_rollouts"
# os.makedirs(out_folder, exist_ok=True)

# save_path = os.path.join(out_folder, "KAE.npz")
# np.savez_compressed(save_path, eval_arrays[None, None, ...])
# print(f"✅ Saved all validation rollouts to {save_path}, shape: {eval_arrays[None, None, ...].shape}")